In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from datetime import datetime, timedelta

# --- ESTILO VISUAL PICPAY ---
display(HTML("""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;700&display=swap');
    .app-container { background: #0A0A0A; font-family: 'Plus Jakarta Sans', sans-serif; border-radius: 25px; padding: 20px; color: white; max-width: 500px; border: 1px solid #333; margin: auto; }
    .picpay-green { color: #21C25E; }
    .btn-picpay { background: #21C25E !important; color: white !important; font-weight: bold !important; border-radius: 12px !important; border: none !important; margin-top: 5px; }
    .card-extrato { background: #181818; padding: 15px; border-radius: 12px; margin: 8px 0; border-left: 5px solid #21C25E; }
    .aviso-juros { background: rgba(255,75,75,0.15); color: #FF4B4B; padding: 10px; border-radius: 10px; font-size: 11px; margin: 10px 0; border: 1px solid #FF4B4B; text-align: center; }
    .score-badge { background: #333; padding: 2px 8px; border-radius: 5px; font-size: 10px; float: right; }
</style>
"""))

class PicPayFinalLocal:
    def __init__(self):
        self.banco_local = {}
        self.out = widgets.Output()
        self.tabs = widgets.Tab(children=[self.ui_contratar(), self.ui_extrato(), self.ui_renegociar()])
        self.tabs.set_title(0, 'Contratar')
        self.tabs.set_title(1, 'Extrato')
        self.tabs.set_title(2, 'Renegociar')

    def validar_cpf(self, change):
        val = "".join(filter(str.isdigit, change['new']))
        self.in_cpf.value = val[:11]

    def ui_contratar(self):
        self.in_nome = widgets.Text(placeholder="Nome Completo")
        self.in_mae = widgets.Text(placeholder="Nome da Mãe")
        self.in_cpf = widgets.Text(placeholder="CPF (11 dígitos)")
        self.in_cpf.observe(self.validar_cpf, names='value')
        self.in_end = widgets.Text(placeholder="Endereço")
        self.in_renda = widgets.FloatText(description="Renda R$:")
        self.in_valor = widgets.FloatText(description="Valor R$:")
        aviso = widgets.HTML("<div class='aviso-juros'>⚠️ JUROS DE 30% APÓS 1 MINUTO</div>")
        btn = widgets.Button(description="AVERBAR EMPRÉSTIMO", layout={'width': '100%'})
        btn.add_class("btn-picpay")
        btn.on_click(self.processar_contratacao)
        return widgets.VBox([self.in_nome, self.in_mae, self.in_cpf, self.in_end, self.in_renda, self.in_valor, aviso, btn])

    def ui_extrato(self):
        self.out_extrato = widgets.Output()
        btn = widgets.Button(description="🔄 ATUALIZAR STATUS", layout={'width': '100%'})
        btn.add_class("btn-picpay")
        btn.on_click(self.renderizar_extrato)
        return widgets.VBox([btn, self.out_extrato])

    def ui_renegociar(self):
        self.in_cpf_acordo = widgets.Text(placeholder="CPF para Renegociar")
        self.out_acordo = widgets.Output()
        btn = widgets.Button(description="BUSCAR DÍVIDAS", layout={'width': '100%'})
        btn.add_class("btn-picpay")
        btn.on_click(self.buscar_acordo)
        return widgets.VBox([self.in_cpf_acordo, btn, self.out_acordo])

    def processar_contratacao(self, b):
        with self.out:
            clear_output()
            cpf = self.in_cpf.value
            if len(cpf) != 11:
                print("❌ CPF Inválido")
                return
            if self.banco_local.get(cpf, {}).get('status') == 'NEGATIVADO':
                display(HTML("<div class='aviso-juros'>❌ BLOQUEIO: CPF Negativado (Baixo Score)</div>"))
                return

            self.banco_local[cpf] = {
                'nome': self.in_nome.value, 'mae': self.in_mae.value, 'renda': self.in_renda.value,
                'valor': self.in_valor.value, 'status': 'ATIVO', 'score': 850,
                'vencimento': datetime.now() + timedelta(minutes=1), 'end': self.in_end.value
            }
            print(f"✅ Empréstimo Aprovado! Score: 850")

    def renderizar_extrato(self, b):
        with self.out_extrato:
            clear_output()
            agora = datetime.now()
            for cpf, info in self.banco_local.items():
                if agora > info['vencimento'] and info['status'] == 'ATIVO':
                    info['status'] = 'NEGATIVADO'
                    info['valor'] *= 1.30
                    info['score'] = 150 # Score cai na negativação

                cor = "#FF4B4B" if info['status'] == 'NEGATIVADO' else "#21C25E"
                display(HTML(f"""
                <div class='card-extrato' style='border-left-color: {cor}'>
                    <span class='score-badge'>Score: {info['score']}</span>
                    <b>{info['nome']}</b><br>
                    <small>Renda: R$ {info['renda']:.2f} | CPF: {cpf}</small><br>
                    <b style='color:{cor}'>R$ {info['valor']:.2f} - {info['status']}</b>
                </div>
                """))

    def buscar_acordo(self, b):
        with self.out_acordo:
            clear_output()
            cpf = self.in_cpf_acordo.value
            if cpf in self.banco_local and self.banco_local[cpf]['status'] == 'NEGATIVADO':
                valor_com_desconto = self.banco_local[cpf]['valor'] * 0.70
                display(HTML(f"<b>Dívida encontrada!</b><br>Valor atual: R$ {self.banco_local[cpf]['valor']:.2f}"))
                display(HTML(f"<p style='color:#21C25E'>Proposta de Acordo: R$ {valor_com_desconto:.2f} (30% OFF)</p>"))

                btn_pagar = widgets.Button(description="PAGAR AGORA", button_style='success')
                btn_pagar.on_click(lambda x: self.quitar_divida(cpf, valor_com_desconto))
                display(btn_pagar)
            else:
                print("Nenhuma dívida para negativação encontrada para este CPF.")

    def quitar_divida(self, cpf, novo_valor):
        self.banco_local[cpf]['status'] = 'ATIVO'
        self.banco_local[cpf]['valor'] = 0
        self.banco_local[cpf]['score'] = 950 # Score sobe após pagar
        self.banco_local[cpf]['vencimento'] = datetime.now() + timedelta(days=365)
        with self.out_acordo:
            clear_output()
            display(HTML("<b style='color:#21C25E'>✅ Dívida Paga! Score subiu para 950. CPF Limpo.</b>"))

# Execução
app = PicPayFinalLocal()
display(HTML("<div class='app-container'><h2 style='text-align:center' class='picpay-green'>PicPay Local Pro</h2>"),
        app.tabs, app.out, HTML("</div>"))

Output()